# 01_prepare_data

Dieses Notebook dokumentiert die Datenaufbereitung der Meerjungfrauenmärchen-Studie.

Ziel ist es, den ursprünglichen LimeSurvey-Export zu bereinigen, zentrale Variablen umzubenennen und strukturierte Analyse-Dateien zu erzeugen. Diese Dateien dienen anschließend als Grundlage für die Reproduktion der zentralen Ergebnisse der Masterarbeit.

Erzeugte Dateien:

- `master_clean_renamed.xlsx`
- `ratings_long.xlsx`
- `compare_long.xlsx`
- `estimation_long.xlsx`


# Bibliotheken laden

In [1]:
import pandas as pd
import numpy as np
import re
from pathlib import Path

## 1. Dateipfade

Hier werden die Pfade für den Rohdatensatz und den Ausgabeordner definiert.

In [2]:
#from google.colab import drive
#drive.mount('/content/drive')

base_dir = Path("") #Pfad eingeben
raw_file = base_dir / "" #Rohdatensatz eingeben
out_dir = base_dir / "data" / "processed"
out_dir.mkdir(parents=True, exist_ok=True)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## 2. Rohdatensatz einlesen

Der ursprüngliche LimeSurvey-Export wird eingelesen und in seiner Grundstruktur überprüft.

In [3]:
df_raw = pd.read_excel(raw_file)

print("Rohdatensatz eingelesen.")
print("Form:", df_raw.shape)
print("\nErste Spalten:")
print(df_raw.columns[:20].tolist())

Rohdatensatz eingelesen.
Form: (160, 605)

Erste Spalten:
['id. Antwort ID', 'submitdate. Datum Abgeschickt', 'lastpage. Letzte Seite', 'startlanguage. Start-Sprache', 'seed. Zufallsstartwert', 'startdate. Datum gestartet', 'datestamp. Datum letzte Aktivität', 'G02Q252. Vielen Dank für Ihr Interesse an dieser Studie.  In dieser Online-Studie untersuchen wir, wie kurze literarische Textausschnitte wahrgenommen und bewertet werden. Im Rahmen der Studie werden Sie gebeten, Textausschnitte zu lesen und Fragen zu Ihrem Leseeindruck zu beantworten.  Erhobene Daten  Im Rahmen der Studie werden folgende Daten erhoben:   \t \tAngaben zu Ihrem Alter \t \t \tAngaben zu Ihrem Geschlecht \t \t \tIhr höchster Bildungsabschluss \t \t \tIhre Muttersprache \t \t \tAngaben zur Häufigkeit des Lesens literarischer Texte und der Leseerfahrung\xa0 \t \t \tIhre Antworten auf die Fragen zu den Textausschnitten \t \t \tAngaben zur Bearbeitungsdauer der Umfrage \t   Es werden keine sensiblen personenbezogenen D

## 3. Datenbereinigung

Es werden alle Fälle ausgeschlossen, die die vorab festgelegten Ausschlusskriterien nicht erfüllen:

- Gesamtbearbeitungszeit unter 900 Sekunden
- mindestens eine Gruppenzeit unter 90 Sekunden

In [4]:
COL_ID = "id. Antwort ID"
COL_GROUP_RAW = "gleichung. {if(is_empty(gleichung), rand(1,2), gleichung)}"
COL_INTERVIEWTIME = "interviewtime. Gesamtzeit"

GROUPTIME_COLS = [
    "groupTime308. Gruppenzeit: Textausschnitte Fragen",
    "groupTime309. Gruppenzeit: Textausschnitte Fragen",
    "groupTime310. Gruppenzeit: Textausschnitte Fragen",
    "groupTime304. Gruppenzeit: Textausschnitte Fragen",
]

required_cols = [COL_GROUP_RAW, COL_INTERVIEWTIME] + GROUPTIME_COLS
missing_cols = [col for col in required_cols if col not in df_raw.columns]
if missing_cols:
    raise ValueError(f"Fehlende benötigte Spalten:\n{missing_cols}")

df_clean = df_raw.copy()

df_clean[COL_INTERVIEWTIME] = pd.to_numeric(df_clean[COL_INTERVIEWTIME], errors="coerce")
for col in GROUPTIME_COLS:
    df_clean[col] = pd.to_numeric(df_clean[col], errors="coerce")
df_clean[COL_GROUP_RAW] = pd.to_numeric(df_clean[COL_GROUP_RAW], errors="coerce")

mask_interviewtime_ok = df_clean[COL_INTERVIEWTIME] >= 900
mask_grouptime_ok = (df_clean[GROUPTIME_COLS] >= 90).all(axis=1)
mask_keep = mask_interviewtime_ok & mask_grouptime_ok

n_raw = len(df_clean)
n_removed_interviewtime = (~mask_interviewtime_ok).sum()
n_removed_grouptime = (~mask_grouptime_ok).sum()
n_removed_total = (~mask_keep).sum()

df_excluded = df_clean.loc[~mask_keep].copy()
df_clean = df_clean.loc[mask_keep].copy()

df_clean["Gruppe"] = df_clean[COL_GROUP_RAW].map({1: "A", 2: "B"})

print("Bereinigung abgeschlossen.")
print(f"Ursprüngliche Fälle: {n_raw}")
print(f"Entfernt wegen Gesamtzeit < 900s: {n_removed_interviewtime}")
print(f"Entfernt wegen mindestens einer Gruppenzeit < 90s: {n_removed_grouptime}")
print(f"Insgesamt entfernte Fälle: {n_removed_total}")
print(f"Verbleibende Fälle: {len(df_clean)}")
print("\nGruppenverteilung:")
print(df_clean["Gruppe"].value_counts(dropna=False))

Bereinigung abgeschlossen.
Ursprüngliche Fälle: 160
Entfernt wegen Gesamtzeit < 900s: 19
Entfernt wegen mindestens einer Gruppenzeit < 90s: 14
Insgesamt entfernte Fälle: 22
Verbleibende Fälle: 138

Gruppenverteilung:
Gruppe
B    69
A    69
Name: count, dtype: int64


## 4. Hilfsfunktionen

Für die weitere Datenaufbereitung werden Hilfsfunktionen definiert, etwa zur Extraktion numerischer Likert-Werte und zur Rekonstruktion von Texttyp und Präferenzrichtung.

In [5]:
def to_numeric_likert(series: pd.Series) -> pd.Series:
    if pd.api.types.is_numeric_dtype(series):
        return series
    s = series.astype(str).str.strip()
    out = s.str.extract(r"^(\d+)", expand=False)
    return pd.to_numeric(out, errors="coerce")


def parse_ab_column(colname: str):
    match = re.match(r"^([AB])(\d+)", colname)
    if not match:
        return None

    group_letter = match.group(1)
    digits = match.group(2)

    if len(digits) < 3:
        return None

    pair_id = int(digits[0])
    text_pos = int(digits[1])

    if len(digits) == 3:
        item_no = int(digits[2])
    else:
        item_no = int(digits[2:])

    return {
        "group_letter": group_letter,
        "pair_id": pair_id,
        "text_pos": text_pos,
        "item_no": item_no,
    }


def parse_v_column(colname: str):
    match = re.match(r"^V([AB])(\d)(\d)(\d+)", colname)
    if not match:
        return None

    return {
        "group_letter": match.group(1),
        "pair_id": int(match.group(2)),
        "comparison_block": int(match.group(3)),
        "question_no": int(match.group(4)),
    }


def get_true_text_type(pair_id: int, text_pos: int) -> str:
    if pair_id in [1, 4]:
        return "human" if text_pos == 1 else "ai"
    elif pair_id in [2, 3]:
        return "ai" if text_pos == 1 else "human"
    return np.nan


def get_preference_target(pair_id: int, scale_value: float) -> str:
    if pd.isna(scale_value):
        return np.nan
    if scale_value == 4:
        return "neutral"

    if pair_id in [1, 4]:
        return "human" if scale_value < 4 else "ai"
    elif pair_id in [2, 3]:
        return "ai" if scale_value < 4 else "human"
    return np.nan

## 5. Zentrale Variablen umbenennen

Zur besseren Lesbarkeit werden zentrale Metadaten und demografische Variablen umbenannt.

In [6]:
rename_map = {
    "id. Antwort ID": "participant_id",
    "submitdate. Datum Abgeschickt": "submit_date",
    "startdate. Datum gestartet": "start_date",
    "datestamp. Datum letzte Aktivität": "last_activity",
    "interviewtime. Gesamtzeit": "interview_time_sec",
    "gleichung. {if(is_empty(gleichung), rand(1,2), gleichung)}": "group_code",
    "Q001. Wie alt sind Sie?": "age",
    "Q002. Was ist Ihr Geschlecht?": "gender",
    "Q003. Was ist ihre Muttersprache?": "native_language",
    "Q003[other]. Was ist ihre Muttersprache? [Sonstiges]": "native_language_other",
    "Q004. Was ist Ihr höchster Schulabschluss?": "education",
    "Q004[other]. Was ist Ihr höchster Schulabschluss? [Sonstiges]": "education_other",
    "Q005. Was studieren Sie aktuell oder was ist Ihr Beruf?": "study_or_job_raw",
    "Q006. Wie häufig lesen Sie literarische Texte (z.B. Romane, Märchen, Erzählungen, ..)?": "reading_frequency_raw",
}

df_master = df_clean.copy().rename(columns=rename_map)
df_master["group"] = df_master["group_code"].map({1: "A", 2: "B"})

print("Master-Datensatz erstellt:", df_master.shape)

Master-Datensatz erstellt: (138, 607)


## 6. ART auswerten

Die Antworten des Author Recognition Tests werden in zusammenfassende Kennwerte überführt.

In [7]:
art_cols = [c for c in df_master.columns if c.startswith("ART")]
print("ART-Spalten:", len(art_cols))

def score_art_row(row):
    correct = 0
    total = 0
    for col in art_cols:
        val = str(row[col]).strip().lower()
        if val in ["", "nan", "none"]:
            continue
        total += 1
        is_real = "[real" in col.lower()
        is_fake = "[fake" in col.lower()
        if is_real and val == "ja":
            correct += 1
        elif is_fake and val == "nein":
            correct += 1
    return pd.Series({
        "ART_correct": correct,
        "ART_total_answered": total,
        "ART_rate": correct / total if total > 0 else np.nan
    })

df_master[["ART_correct", "ART_total_answered", "ART_rate"]] = df_master.apply(score_art_row, axis=1)

ART-Spalten: 76


## 7. Einzelbewertungen in ein Long-Format überführen

Die Einzelbewertungen der Textausschnitte werden aus der LimeSurvey-Struktur in ein auswertbares Long-Format überführt.

In [8]:
ab_cols = [c for c in df_master.columns if re.match(r"^[AB]\d", c)]
print("Einzelbewertungs-Spalten:", len(ab_cols))

long_ratings = []

for col in ab_cols:
    parsed = parse_ab_column(col)
    if parsed is None:
        continue

    tmp = df_master[["participant_id", "group", col]].copy()
    tmp = tmp.rename(columns={col: "raw_response"})

    tmp["column_name"] = col
    tmp["group_letter_from_col"] = parsed["group_letter"]
    tmp["pair_id"] = parsed["pair_id"]
    tmp["text_pos"] = parsed["text_pos"]
    tmp["item_no"] = parsed["item_no"]
    tmp["text_type"] = tmp.apply(lambda x: get_true_text_type(x["pair_id"], x["text_pos"]), axis=1)

    tmp = tmp[tmp["group"] == tmp["group_letter_from_col"]].copy()
    tmp = tmp[tmp["raw_response"].notna()].copy()
    tmp = tmp[tmp["raw_response"].astype(str).str.strip() != ""].copy()

    if parsed["item_no"] == 9:
        tmp["response_num"] = np.nan
        tmp["response_text"] = tmp["raw_response"]
    else:
        tmp["response_num"] = to_numeric_likert(tmp["raw_response"])
        tmp["response_text"] = np.nan

    long_ratings.append(tmp)

ratings_long = pd.concat(long_ratings, ignore_index=True)

ratings_long = ratings_long[
    [
        "participant_id", "group", "column_name", "pair_id", "text_pos",
        "text_type", "item_no", "raw_response", "response_num", "response_text"
    ]
].copy()

print("ratings_long:", ratings_long.shape)
ratings_long.head()

Einzelbewertungs-Spalten: 368
ratings_long: (10585, 10)


,participant_id,group,column_name,pair_id,text_pos,text_type,item_no,raw_response,response_num,response_text
0,8,A,A111. Textausschnitt 1: Die Sonne war gerade...,1,1,human,1,7 – stimme voll und ganz zu,7.0,NaN
1,9,A,A111. Textausschnitt 1: Die Sonne war gerade...,1,1,human,1,2 – stimme weitgehend nicht zu,2.0,NaN
2,13,A,A111. Textausschnitt 1: Die Sonne war gerade...,1,1,human,1,6 – stimme weitgehend zu,6.0,NaN
3,15,A,A111. Textausschnitt 1: Die Sonne war gerade...,1,1,human,1,7 – stimme voll und ganz zu,7.0,NaN
4,16,A,A111. Textausschnitt 1: Die Sonne war gerade...,1,1,human,1,7 – stimme voll und ganz zu,7.0,NaN


## 8. Vergleichsfragen in ein Long-Format überführen

In [9]:
v_cols = [c for c in df_master.columns if re.match(r"^V[AB]\d", c)]
print("Vergleichsfragen-Spalten:", len(v_cols))

long_compare = []

for col in v_cols:
    parsed = parse_v_column(col)
    if parsed is None:
        continue

    tmp = df_master[["participant_id", "group", col]].copy()
    tmp = tmp.rename(columns={col: "raw_response"})

    tmp["column_name"] = col
    tmp["group_letter_from_col"] = parsed["group_letter"]
    tmp["pair_id"] = parsed["pair_id"]
    tmp["comparison_block"] = parsed["comparison_block"]
    tmp["question_no"] = parsed["question_no"]

    tmp = tmp[tmp["group"] == tmp["group_letter_from_col"]].copy()
    tmp = tmp[tmp["raw_response"].notna()].copy()
    tmp = tmp[tmp["raw_response"].astype(str).str.strip() != ""].copy()

    if parsed["question_no"] == 2:
        tmp["response_num"] = np.nan
        tmp["response_text"] = tmp["raw_response"]
        tmp["preference_target"] = np.nan
    else:
        tmp["response_num"] = to_numeric_likert(tmp["raw_response"])
        tmp["response_text"] = np.nan
        if parsed["question_no"] in [1, 4, 5]:
            tmp["preference_target"] = tmp.apply(
                lambda x: get_preference_target(x["pair_id"], x["response_num"]),
                axis=1
            )
        else:
            tmp["preference_target"] = np.nan

    long_compare.append(tmp)

compare_long = pd.concat(long_compare, ignore_index=True)

compare_long = compare_long[
    [
        "participant_id", "group", "column_name", "pair_id", "question_no",
        "raw_response", "response_num", "response_text", "preference_target"
    ]
].copy()

print("compare_long:", compare_long.shape)
compare_long.head()

Vergleichsfragen-Spalten: 40
compare_long: (2592, 9)


,participant_id,group,column_name,pair_id,question_no,raw_response,response_num,response_text,preference_target
0,8,A,VA131. Welcher der beiden Textausschnitte wirk...,1,1,4 – beide Textausschnitte gleich stark,4.0,NaN,neutral
1,9,A,VA131. Welcher der beiden Textausschnitte wirk...,1,1,7 – nur Textausschnitt 2,7.0,NaN,ai
2,13,A,VA131. Welcher der beiden Textausschnitte wirk...,1,1,2 – hauptsächlich Textausschnitt 1,2.0,NaN,human
3,15,A,VA131. Welcher der beiden Textausschnitte wirk...,1,1,4 – beide Textausschnitte gleich stark,4.0,NaN,neutral
4,16,A,VA131. Welcher der beiden Textausschnitte wirk...,1,1,4 – beide Textausschnitte gleich stark,4.0,NaN,neutral


## 9. Einschätzungsfragen in ein Long-Format überführen

In [10]:
estimation_cols = [
    c for c in df_master.columns
    if re.match(r"^(M|K|KI)\d+\.", c)
]

comment_cols = [
    c for c in df_master.columns
    if "[comment]" in c and re.match(r"^(M|K|KI)\d+", c)
]

print("Einschätzungsfragen:", len(estimation_cols))
print("Kommentarspalten dazu:", len(comment_cols))

records = []

for col in estimation_cols:
    base_match = re.match(r"^(M|K|KI)(\d+)\.", col)
    if not base_match:
        continue

    prefix = base_match.group(1)
    text_id = int(base_match.group(2))
    true_type = "human" if prefix == "M" else "ai"

    comment_col = None
    for cc in comment_cols:
        if cc.startswith(f"{prefix}{text_id}[comment]"):
            comment_col = cc
            break

    use_cols = ["participant_id", "group", col]
    if comment_col is not None:
        use_cols.append(comment_col)

    tmp = df_master[use_cols].copy()
    tmp = tmp.rename(columns={col: "raw_response"})
    if comment_col is not None:
        tmp = tmp.rename(columns={comment_col: "comment"})

    tmp["column_name"] = col
    tmp["text_id"] = text_id
    tmp["true_type"] = true_type

    tmp = tmp[tmp["raw_response"].notna()].copy()
    tmp = tmp[tmp["raw_response"].astype(str).str.strip() != ""].copy()

    resp = tmp["raw_response"].astype(str).str.strip().str.lower()

    tmp["predicted_type"] = pd.Series(index=tmp.index, dtype="object")
    tmp.loc[resp.str.contains("ki", na=False), "predicted_type"] = "ai"
    tmp.loc[resp.str.contains("mensch", na=False), "predicted_type"] = "human"

    tmp["correct"] = (tmp["predicted_type"] == tmp["true_type"]).astype(float)

    records.append(tmp)

estimation_long = pd.concat(records, ignore_index=True)

keep_cols = [
    "participant_id", "group", "column_name", "text_id",
    "true_type", "raw_response", "predicted_type", "correct"
]
if "comment" in estimation_long.columns:
    keep_cols.append("comment")

estimation_long = estimation_long[keep_cols].copy()

print("estimation_long:", estimation_long.shape)
estimation_long.head()

Einschätzungsfragen: 10
Kommentarspalten dazu: 10
estimation_long: (552, 9)


,participant_id,group,column_name,text_id,true_type,raw_response,predicted_type,correct,comment
0,8,A,M1. O wieviel freudiger brauchte nun der junge...,1,human,KI-generiert,ai,0.0,NaN
1,9,A,M1. O wieviel freudiger brauchte nun der junge...,1,human,KI-generiert,ai,0.0,NaN
2,16,A,M1. O wieviel freudiger brauchte nun der junge...,1,human,Von einem Menschen geschrieben,human,1.0,Die ganzen sprachlichen Ausschmückungen wirken...
3,23,A,M1. O wieviel freudiger brauchte nun der junge...,1,human,KI-generiert,ai,0.0,NaN
4,24,A,M1. O wieviel freudiger brauchte nun der junge...,1,human,Von einem Menschen geschrieben,human,1.0,"Sehr schwierig zu entscheiden, aber ich glaube..."


## 10. Offene Antworten separat ausleiten

In [11]:
open_item9_long = ratings_long[ratings_long["item_no"] == 9].copy()
open_compare2_long = compare_long[compare_long["question_no"] == 2].copy()

print("open_item9_long:", open_item9_long.shape)
print("open_compare2_long:", open_compare2_long.shape)

open_item9_long: (373, 10)
open_compare2_long: (384, 9)


## 11. Plausibilitätschecks

In [12]:
print("--- Plausibilitätschecks ---")
print("ratings_long Gruppen:")
print(ratings_long["group"].value_counts())

print("\nratings_long Texttypen:")
print(ratings_long["text_type"].value_counts())

print("\nratings_long Items:")
print(ratings_long["item_no"].value_counts().sort_index())

print("\ncompare_long Fragen:")
print(compare_long["question_no"].value_counts().sort_index())

print("\nestimation_long true_type:")
print(estimation_long["true_type"].value_counts())

print("\nestimation_long mittlere Korrektheit:")
print(estimation_long["correct"].mean())

--- Plausibilitätschecks ---
ratings_long Gruppen:
group
B    5294
A    5291
Name: count, dtype: int64

ratings_long Texttypen:
text_type
human    5300
ai       5285
Name: count, dtype: int64

ratings_long Items:
item_no
1     1104
2     1104
3     1104
4     1104
5     1104
6     1104
7     1104
8     1104
9      373
10    1104
11     276
Name: count, dtype: int64

compare_long Fragen:
question_no
1    552
2    384
3    552
4    552
5    552
Name: count, dtype: int64

estimation_long true_type:
true_type
ai       282
human    270
Name: count, dtype: int64

estimation_long mittlere Korrektheit:
0.6322463768115942


## 12. Dateien speichern

Die erzeugten Dateien werden als Grundlage für das zweite Notebook gespeichert.

In [13]:
df_master.to_excel(out_dir / "master_clean_renamed.xlsx", index=False)
ratings_long.to_excel(out_dir / "ratings_long.xlsx", index=False)
compare_long.to_excel(out_dir / "compare_long.xlsx", index=False)
estimation_long.to_excel(out_dir / "estimation_long.xlsx", index=False)

# Offene Dateien nur speichern, wenn du sie intern brauchst
open_item9_long.to_excel(out_dir / "open_item9_long.xlsx", index=False)
open_compare2_long.to_excel(out_dir / "open_compare2_long.xlsx", index=False)

print("Dateien gespeichert in:", out_dir)

Dateien gespeichert in: /content/drive/MyDrive/Masterarbeit /Masterarbeit/Analyse/Daten/data/processed
